Until now i have learn  in a superficial way how the tokenization  vectorization and embedding process work  and how they differ between them. now i will writte code of a vectorization layer  and then in the other cell a  embeding layer (neural and no neural methods of vectorization )



In [4]:
#classical tf idf  vectorization|
from sklearn.feature_extraction.text import TfidfVectorizer
print(TfidfVectorizer)

import numpy as np
docs= {"the king rules the kingdom","the queen governs the empire","a man and woman are leaders"}
embeder=TfidfVectorizer()
X=embeder.fit_transform(docs)
print("TF-IDF vector shape:", X.shape)
print("Example vector (first doc):\n", X.toarray()[0])


<class 'sklearn.feature_extraction.text.TfidfVectorizer'>
TF-IDF vector shape: (3, 12)
Example vector (first doc):
 [0.         0.         0.43381609 0.43381609 0.         0.
 0.         0.         0.43381609 0.         0.65985664 0.        ]


In [7]:
#if running in colab server
#!pip install gensim
!pip install mpl_toolkits

ERROR: Could not find a version that satisfies the requirement mpl_toolkits (from versions: none)
ERROR: No matching distribution found for mpl_toolkits


remind that cosine of 0 is 1 and cosine of 180 is -1 so a value of or near to 1 means veri similar . -1 almost oposite, and 0 no relationship. now with the original corpora i got  values of 0.01 for king and man and 0.2 for queen and man, now that could have been because the small corpora, here i augmented te corpora, eith tat the similarity man king impruve but king queen decrese. using diferent parameter for the model



In [12]:
# Word2Vec embeddings
from gensim.models import Word2Vec
import numpy as np
from numpy import dot
from numpy.linalg import norm
docs = {
    "the king rules the kingdom",
    "the queen governs the empire",
    "a man and woman are leaders",
    "the king and the queen lead together",
    "the queen and the king are rulers of the land",
    "the man is loyal to the king",
    "the woman supports the queen in the empire",
    "the king and the man fight for the kingdom",
    "the queen and the woman protect the empire",
    "the king is married to the queen",
    "the man and the woman serve the royal family",
    "the kingdom is ruled by the king and the queen",
    "the empire is governed by the queen and the king",
    "the king is a strong man",
    "the queen is a wise woman"
}

# Tokenize each document properly
tokens = [doc.lower().split() for doc in docs]  # <-- () are needed!

# Train Word2Vec model the window parameter makes mee think this is a convolutional architecture
model = Word2Vec(tokens, vector_size=30, window=3, min_count=1, workers=2)

# Print vector info
print("\nWord2Vec vector shape:", model.wv["king"].shape)
print("Embedding for 'king':\n", model.wv["king"])

# Define cosine similarity
def cosineSim(a: np.ndarray, b: np.ndarray) -> float:
    """
    a and b are embeddings
    """
    return dot(a, b) / (norm(a) * norm(b))

# Compare words semantically
print("Cosine similarity between king and man:",
      cosineSim(model.wv["king"], model.wv["man"]))
print("Cosine similarity between king and queen:",
      cosineSim(model.wv["king"], model.wv["queen"]))




Word2Vec vector shape: (30,)
Embedding for 'king':
 [-2.51820870e-02 -3.27238254e-03  3.20830867e-02 -2.46952344e-02
 -7.78554194e-03 -6.27030199e-03  2.72302795e-02 -1.96377914e-02
 -1.21699188e-04 -1.55937634e-02 -3.18442397e-02  1.66883729e-02
 -2.92591173e-02 -1.47894090e-02 -3.57330900e-05 -8.89667834e-04
 -2.52749790e-02  3.21466029e-02  1.65827926e-02  3.08366176e-02
 -2.71417089e-02  1.48385670e-02 -1.37442360e-02  2.87719769e-03
  2.82562003e-02 -1.48470299e-02  1.52988285e-02 -2.26703845e-02
 -1.19394315e-02  3.11786216e-02]
Cosine similarity between king and man: 0.30010256
Cosine similarity between king and queen: -0.0580755


frecuenzy based methods give a higth dimmenssional sparse representation wich is problematic specially because of the distances problem
word to vec methods give a dense low dimentionality representation and the semantic meanning is shown by the cosine of the angle between 2 vectores

In [6]:
import gensim.downloader as api
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from mpl_toolkits.plot3d import Axes3D
model= api.load("word2Vec-google-news-300")


ModuleNotFoundError: No module named 'mpl_toolkits.plot3d'

In [13]:
len (model['bird'])
v_king=model['king']
v_queen=model['queen']
v_man=model['man']
v_woman=model['woman']
v_result=v_king - v_man + v_woman
#print(f"now the result of the operation is {v_result} and  queen is {v_queen}")
vectors_to_plot=[v_king,v_man,v_queen,v_woman,v_result]
words_to_plot = ['king','man','queen','woman']
pca=PCA(n_components=3)
pca_fitted=pca.fit_transform(vectors_to_plot)
#this next line only works if  words to plot and vectors to plot have the same order
word_cords = {word:pca_fitted[i] for i,word in enumerate(words_to_plot)}
def cos (a,b):
    return cosine_similarity([a,b])[0,0] #posicion the la matriz
fig=plt.figure(figsize=(10,5))
ax=fig.add_subplot(111,projection='3d')
# Scatter + labels
for word, vec in word_cords.items():
    ax.scatter(vec[0], vec[1], vec[2], s=50)
    ax.text(vec[0], vec[1], vec[2], word, fontsize=10)

# Retrieve coordinates
result = word_cords['result']
king = word_cords['king']
queen = word_cords['queen']

# Draw lines connecting result↔king, result↔queen, and king↔queen
ax.plot([result[0], king[0]], [result[1], king[1]], [result[2], king[2]], color='r', linestyle='--', label='result-king')
ax.plot([result[0], queen[0]], [result[1], queen[1]], [result[2], queen[2]], color='g', linestyle='--', label='result-queen')
ax.plot([king[0], queen[0]], [king[1], queen[1]], [king[2], queen[2]], color='b', linestyle='--', label='king-queen')

# Aesthetic tweaks
ax.set_title("Word2Vec Analogy Relationships in 3D (PCA projection)")
ax.legend()
plt.show()


TypeError: 'Word2Vec' object is not subscriptable

## MULTI LAYER PERCEPTRON 
DOWN WE IMPORT  from pytorch the modules:
- nn: that have the basics for neural networks
- optim: that have the optimizer
- functional: have the activation cunctions inside
then we create our class  inheratin from nn module  and pass as parameters the dimention of all layers inside.

In [19]:
import torch
import torch.nn as nn
import torch.optim as opt
import torch.nn.functional as fc
import time 
device=torch.device('cuda'if torch.cuda.is_available() else 'cpu')
print ('usando',device)
class MLP (nn.Module):
    def __init__(self,input,hiden,output):
        super (MLP,self).__init__()
        self.fl=nn.Linear(input,hiden)        
        self.f2=nn.Linear(hiden,output)
    def forward (self,x):
        x=fc.relu(self.f1(x))
        x=self.f2(x)
        return x
start=time.time()
model=MLP(10,32,1)
criterion=nn.MSELoss()
optimizer=opt.Adam(model.parameters(),lr=0.01)

#aca creamos datos ficticios  vectoes de 10 y 1 dimencion
input=torch.randn(200000,10)
labels=torch.randn(200000,1)


usando cuda


wee created  a dataset with 200 thousand  samples using torch random module now  we will train  that model the steps to train the model ate as follows 

1) put it in train mode: important for droput and batch norm layers that  behave diferently depending 
2) reset gradients(compass)
3) forward pass of the data into the model
4) compute loss 
5) back propagation of the error to know  acountability
6) weigth update 

In [22]:
start=time.time()
model.train()
optimizer.zero_grad()
output=model.forward(input)
loss=criterion(output,labels)
loss.backwards()
optimizer.step
end=time.time()
lap=end-start
print(loss.item())
print(f'it took {lap}')

AttributeError: 'MLP' object has no attribute 'f1'